# Figure 4 — An independent simulator: dyngen (S21b, 8 time points; inset S21, 3 time points)

Sources: `benchmarks/dyngen/s21b/scores.csv` (per dataset × operator × drift seed: fork AUC,
median rotational index max|Im λ|/max|λ|, fraction Re λ₁ > 0 by truth region),
`benchmarks/dyngen/s21b/part_a_T.csv` (fork-cell counts at T = 8),
`benchmarks/dyngen/s21b/RESULT_S21b.md` (report-only table of the all-cell linear fraction),
`benchmarks/dyngen/s21/scores.csv` (T = 3 fork counts). Truth used for scoring only.

In [1]:
from _style import *
import re

prov = Provenance("Fig4", notebook="notebooks/fig4_dyngen.ipynb")
sc = pd.read_csv(prov.use("benchmarks/dyngen/s21b/scores.csv", "a,b,c", "S21b"))
pa = pd.read_csv(prov.use("benchmarks/dyngen/s21b/part_a_T.csv", "d", "S21b"))
res_md = prov.use("benchmarks/dyngen/s21b/RESULT_S21b.md", "c", "S21b", note="report-only linear_all table").read_text()
s21 = pd.read_csv(prov.use("benchmarks/dyngen/s21/scores.csv", "d", "S21"))
V = prov.value

# all-cell fraction on linear backbones (report-only table in RESULT_S21b.md)
lin_all = [(m.group(1), int(m.group(2)), float(m.group(3))) for m in
           re.finditer(r"\| (linear_s\d_T8) \| (\d) \| linear_all \| 5000 \| ([0-9.]+) \|", res_md)]
assert len(lin_all) == 9, lin_all
V("linear_all_fraction_max", max(x[2] for x in lin_all)); V("linear_all_n_rows", len(lin_all))

fig = plt.figure(figsize=(DOUBLE_COL, 2.7))
gs = fig.add_gridspec(1, 4, width_ratios=[1.0, 0.95, 1.55, 0.75], wspace=0.78)

In [2]:
ax = fig.add_subplot(gs[0, 0])
bif = sc[sc.backbone == "bifurcating"]
dsets = sorted(bif.dataset.unique())
for i, ds in enumerate(dsets):
    f = bif[(bif.dataset == ds) & (bif.operator == "flow")]
    ax.scatter(i - 0.12 + np.array([-0.06, 0, 0.06]), f.fork_auc, color=C_FLOW, s=12, zorder=3)
    s = bif[(bif.dataset == ds) & (bif.operator == "score")]
    ax.scatter([i + 0.15], s.fork_auc, color=C_SCORE, marker="s", s=12, zorder=3)
    V(f"a_flow_auc_{ds}_min", f.fork_auc.min()); V(f"a_flow_auc_{ds}_max", f.fork_auc.max())
    V(f"a_score_auc_{ds}", float(s.fork_auc.iloc[0]))
ax.axhline(0.85, color=C_BASE, ls=":", lw=0.6); ax.axhline(0.5, color=C_BASE, lw=0.4)
ax.set_xticks(range(len(dsets))); ax.set_xticklabels([d.replace("bifurcating_v2_", "").replace("_T8", "") for d in dsets])
ax.set_xlabel("bifurcating dataset"); ax.set_ylabel("fork AUC"); ax.set_ylim(0, 1.05)
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=C_FLOW, ms=3.5, label="flow J\n(drift seeds 0–2)"),
                   plt.Line2D([], [], marker="s", ls="", color=C_SCORE, ms=3.5, label="score J")], loc="lower left", fontsize=6, handletextpad=0.2, borderaxespad=0.2)
panel_letter(ax, "a", x=-0.3)
n_pass = sum(bif[(bif.dataset == d) & (bif.operator == "flow")].fork_auc.min() >= 0.85 for d in dsets)
V("a_n_bifurcations_all_seeds_ge_0_85", int(n_pass)); V("a_n_bifurcations", len(dsets))
prov.panel("a", "Fork AUC per bifurcating dataset: flow J for drift seeds 0-2, score J (one fit)")

In [3]:
ax = fig.add_subplot(gs[0, 1])
fl = sc[sc.operator == "flow"].copy()
fl["pair"] = fl.dataset.str.extract(r"_(s\d)_T8")[0]
n_cyc_pass = 0
for i, p in enumerate(["s0", "s1", "s2"]):
    c = fl[(fl.backbone == "cycle") & (fl.pair == p)].sort_values("drift_seed")
    l = fl[(fl.backbone == "linear") & (fl.pair == p)].sort_values("drift_seed")
    for (cv, lv) in zip(c.median_rotation, l.median_rotation):
        ax.plot([i - 0.15, i + 0.15], [cv, lv], color="#BBBBBB", lw=0.6, zorder=1)
    ax.scatter(i - 0.15 + np.zeros(3), c.median_rotation, color=C_FLOW, s=12, zorder=3)
    ax.scatter(i + 0.15 + np.zeros(3), l.median_rotation, color="white", edgecolor=C_FLOW, s=12, zorder=3)
    passed = all(np.asarray(c.median_rotation) > np.asarray(l.median_rotation))
    n_cyc_pass += passed
    V(f"b_cycle_{p}_min", c.median_rotation.min()); V(f"b_cycle_{p}_max", c.median_rotation.max())
    V(f"b_linear_{p}_min", l.median_rotation.min()); V(f"b_linear_{p}_max", l.median_rotation.max())
V("b_n_cycles_pass_all_seeds", int(n_cyc_pass))
sr = sc[(sc.operator == "score")].median_rotation
V("b_score_rotation_max", float(sr.max()))
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["s0", "s1", "s2"]); ax.set_xlabel("seed (cycle ● vs linear ○)")
ax.set_ylabel("median rotational index"); ax.set_ylim(0, 0.45)
panel_letter(ax, "b", x=-0.3)
prov.panel("b", "Median rotational index of flow J on cycle vs linear backbone, per dataset seed and drift seed")

In [4]:
ax = fig.add_subplot(gs[0, 2])
regions = [("fraction_positive_early", "pre-branch /\nearly"), ("fraction_positive_fork", "fork"),
           ("fraction_positive_committed", "committed")]
rcol = [OKABE["skyblue"], C_FLOW, OKABE["green"]]
order = dsets + ["linear_s0_T8", "linear_s1_T8", "linear_s2_T8"]
for i, ds in enumerate(order):
    f = fl[fl.dataset == ds]
    for j, (col, lab) in enumerate(regions):
        v = f[col].values
        if np.all(np.isnan(v)):
            continue
        x = i + (j - 1) * 0.25
        ax.bar(x, np.nanmedian(v), width=0.23, color=rcol[j], label=lab if i == 0 else None)
        ax.scatter(x + np.array([-0.06, 0, 0.06]), v, s=3, color="black", zorder=3, lw=0)
for ds in order:
    f = fl[fl.dataset == ds]
    for col, _ in regions:
        if f[col].notna().any():
            V(f"c_{ds}_{col}_min", float(f[col].min())); V(f"c_{ds}_{col}_max", float(f[col].max()))
ax.set_xticks(range(len(order)))
ax.set_xticklabels([d.replace("bifurcating_v2_", "bif ").replace("_T8", "").replace("linear_", "lin ") for d in order], fontsize=5.8, rotation=45, ha="right")
ax.set_ylabel("fraction of cells with Re λ₁ > 0 (flow J)"); ax.set_ylim(0, 1.05)
ax.legend(fontsize=6, loc="center right", bbox_to_anchor=(1.03, 0.68), ncol=1, handlelength=1.2, handletextpad=0.4)
ax.text(4.3, 0.1, "0 in every region\nand seed", ha="center", fontsize=6, color="#444444")
panel_letter(ax, "c", x=-0.15)
prov.panel("c", "Fraction of cells with Re lambda1 > 0 by truth region (flow J; bar = median over drift seeds, dots = seeds)")

# inset d — fork-cell counts at 3 vs 8 time points
ins = fig.add_subplot(gs[0, 3])
t3 = s21[(s21.backbone == "bifurcating") & (s21.operator == "flow") & (s21.drift_seed.astype(str) == "0")].set_index("dataset").n_fork
t8 = pa[pa.backbone == "bifurcating"].set_index("source_key").fork_cells
keys = sorted(t8.index)
xx = np.arange(len(keys))
ins.bar(xx - 0.18, [t3.get(k, np.nan) for k in keys], width=0.34, color=C_BASE, label="T = 3 (S21)")
ins.bar(xx + 0.18, [t8[k] for k in keys], width=0.34, color=C_FLOW, label="T = 8 (S21b)")
ins.set_xticks(xx); ins.set_xticklabels([k.replace("bifurcating_v2_", "") for k in keys])
ins.set_ylabel("cells in fork region"); ins.set_xlabel("bifurcating dataset")
for x_, k in zip(xx, keys):
    ins.text(x_ - 0.18, t3.get(k, 0) + 10, str(int(t3.get(k, 0))), ha="center", fontsize=5)
panel_letter(ins, "d", x=-0.45)
ins.legend(fontsize=6, loc="upper left", handlelength=1, borderaxespad=0.1)
ins.set_ylim(0, 1050)
for k in keys:
    V(f"d_fork_cells_T3_{k}", int(t3.get(k))); V(f"d_fork_cells_T8_{k}", int(t8[k]))
prov.panel("d", "Fork-region cell counts at 3 (S21) vs 8 (S21b) synchronised time points")
save(fig, "Fig4")

[save] main/Fig4: 160.4 x 64.2 mm, PNG 3788 x 1517 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig4.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig4.png'))

In [5]:
v = prov.values
R = lambda a, b: rng_str(v[a], v[b])
dk = [d for d in dsets]
prov.legend(f"""
**Figure 4 | An independent simulator (dyngen; S21b, pre-registered).** Nine dyngen datasets (bifurcating seeds 10/16/18,
cycle s0–s2, linear s0–s2; 5,000 cells, 1,000 genes), re-sampled from the saved S21 simulations at eight synchronised time points;
same untuned settings as S14's reference flow J (PCA-15, entropic OT, MeasuredDrift drift seeds 0–2) and S10 score J; truth read
only after all fits. **a**, Fork AUC (λ₁, fork vs committed cells): flow J {R(f'a_flow_auc_{dk[0]}_min', f'a_flow_auc_{dk[0]}_max')},
{R(f'a_flow_auc_{dk[1]}_min', f'a_flow_auc_{dk[1]}_max')} and {R(f'a_flow_auc_{dk[2]}_min', f'a_flow_auc_{dk[2]}_max')} (inverted) on
s10, s16, s18; score J {f2(v[f'a_score_auc_{dk[0]}'])}, {f2(v[f'a_score_auc_{dk[1]}'])}, {f2(v[f'a_score_auc_{dk[2]}'])}.
Gate F1 (≥ 0.85) holds on {v['a_n_bifurcations_all_seeds_ge_0_85']} of {v['a_n_bifurcations']} datasets (gate failed).
**b**, Median rotational index max|Im λ|/max|λ| of flow J, cycle (filled) vs linear (open) per dataset seed and drift seed (lines pair
the same drift seed): cycle s1 {R('b_cycle_s1_min', 'b_cycle_s1_max')} and s2 {R('b_cycle_s2_min', 'b_cycle_s2_max')} vs linear
{rng_str(min(v['b_linear_s1_min'], v['b_linear_s2_min']), max(v['b_linear_s1_max'], v['b_linear_s2_max']))}; cycle s0
{R('b_cycle_s0_min', 'b_cycle_s0_max')} vs linear s0 {R('b_linear_s0_min', 'b_linear_s0_max')} (gate F2 failed: {v['b_n_cycles_pass_all_seeds']} of 3
cycles pass on every seed). Score J is symmetric (rotational index {f2(v['b_score_rotation_max'])}).
**c**, Fraction of cells with Re λ₁ > 0 (flow J) by truth region (report only): on linear backbones the fraction is 0 in the early and
committed regions for every seed, and 0 over all 5,000 cells (maximum {f2(v['linear_all_fraction_max'], 4)} over {v['linear_all_n_rows']} dataset × seed
rows, RESULT_S21b.md). **d**, Cells in the fork region at three time points (S21) vs eight (S21b):
s10 {v['d_fork_cells_T3_bifurcating_v2_s10']} vs {v['d_fork_cells_T8_bifurcating_v2_s10']}, s16 {v['d_fork_cells_T3_bifurcating_v2_s16']} vs
{v['d_fork_cells_T8_bifurcating_v2_s16']}, s18 {v['d_fork_cells_T3_bifurcating_v2_s18']} vs {v['d_fork_cells_T8_bifurcating_v2_s18']}.
Diagnosis of the inverted dataset (s18): pending.
""")
prov.write()